# Fase 3: líneas base y evaluación de origen móvil

Este notebook deja constancia de cómo se evalúan las **líneas base** (el suelo que tendrán que
superar los modelos de la fase 4) y qué resultados dan sobre las series elegibles de la fase 2.

Todas las celdas llaman a funciones del paquete `prevision` y no tienen lógica propia, así que el
notebook y el pipeline (`python -m prevision.pipeline`) no pueden divergir. La última celda lo
comprueba.

**Las tres líneas base** (cada una es una función `historia -> previsión del curso siguiente`):

- `naive`: el último valor.
- `drift`: último valor más la pendiente media por curso entre el primer y el último curso de la historia.
- `moving_average`: media de los últimos 3 cursos.

**El marco de origen móvil.** Para cada serie y cada origen `t`, el modelo ve solo los cursos con dato
hasta `t` y prevé `t+1`. Un par (serie, origen) existe si la historia tiene al menos `MIN_TRAIN`
cursos con dato y el curso `t+1` tiene dato. La historia se corta antes de llamar al modelo, de modo
que ningún dato posterior a `t` puede influir en la previsión (hay un test que lo comprueba).

In [1]:
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display

from prevision.config import LINKS_FILE, MA_WINDOW, MIN_TRAIN, RAW_FILE
from prevision.evaluation import (
    add_size_tier,
    compare_to_naive,
    compute_metrics,
    metrics_report,
    rolling_origin_forecasts,
)
from prevision.pipeline import build_series, evaluate_baselines, run_pipeline

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Los ficheros del notebook van a una carpeta temporal: no tocan data/processed/.
out_dir = Path(tempfile.mkdtemp())
panel = run_pipeline(RAW_FILE, out_dir)
_, series, quality = build_series(panel, LINKS_FILE, out_dir)
forecasts, metrics = evaluate_baselines(series, quality, out_dir)
print(f"Parámetros: MIN_TRAIN={MIN_TRAIN}, MA_WINDOW={MA_WINDOW}")

Parámetros: MIN_TRAIN=4, MA_WINDOW=3


## 1. Pares evaluados

Entran solo las series **elegibles** de la fase 2. Una serie elegible con exactamente `MIN_TRAIN`
cursos no da ningún par: necesita un curso más como objetivo.

In [2]:
naive_pairs = forecasts[forecasts["model"] == "naive"]
print(
    f"{forecasts['series_id'].nunique()} series con pares, de {int(quality['eligible'].sum())} elegibles; "
    f"{len(naive_pairs)} pares por modelo"
)
display(naive_pairs.groupby("target_year").size().rename("pares").to_frame().T)

110 series con pares, de 112 elegibles; 761 pares por modelo


target_year,2019,2020,2021,2022,2023,2024,2025
pares,107,107,108,109,110,110,110


## 2. Métricas

Sobre todos los pares de cada modelo:

- **MAE**: error absoluto medio, en estudiantes.
- **WAPE** = suma de |error| / suma de matrícula real. Es el error relativo del conjunto: pesan más
  los centros grandes.
- **MAPE** = media de |error| / matrícula real. Es el error relativo medio por par: todos los
  centros pesan igual.
- **Sesgo**: error medio con signo (previsión menos real). Negativo = el modelo infravalora.
- `*_vs_naive`: la métrica del modelo dividida entre la del ingenuo, menos 1. Negativo = mejor que el
  ingenuo.

El curso 2025-26 es **provisional**: se informa con todos los pares y sin los pares cuyo objetivo es
ese curso.

In [3]:
cols = ["model", "n_pairs", "mae", "wape", "mape", "bias", "mae_vs_naive", "wape_vs_naive", "mape_vs_naive"]
for scope in ("todos", "sin_provisional"):
    print(f"Pares: {scope}")
    display(metrics[(metrics["scope"] == scope) & metrics["target_year"].isna()][cols].round(3))

Pares: todos


,model,n_pairs,mae,wape,mape,bias,mae_vs_naive,wape_vs_naive,mape_vs_naive
0,drift,761,20.744,0.044,0.047,-5.109,-0.083,-0.083,-0.061
1,moving_average,761,37.172,0.078,0.084,-24.107,0.643,0.643,0.657
2,naive,761,22.629,0.048,0.051,-12.648,0.000,0.000,0.000


Pares: sin_provisional


,model,n_pairs,mae,wape,mape,bias,mae_vs_naive,wape_vs_naive,mape_vs_naive
24,drift,651,20.299,0.043,0.047,-5.323,-0.085,-0.085,-0.052
25,moving_average,651,35.965,0.077,0.080,-22.895,0.622,0.622,0.612
26,naive,651,22.174,0.047,0.050,-12.422,0.000,0.000,0.000


### WAPE por curso objetivo

Si un modelo solo gana en un curso concreto, la mejora no es robusta.

In [4]:
by_year = metrics[(metrics["scope"] == "todos") & metrics["target_year"].notna()]
display(by_year.pivot(index="target_year", columns="model", values="wape").round(3))

model,drift,moving_average,naive
target_year,,,
2019.0,0.037,0.065,0.040
2020.0,0.043,0.069,0.043
2021.0,0.039,0.066,0.043
2022.0,0.050,0.083,0.054
2023.0,0.047,0.084,0.049
2024.0,0.043,0.091,0.053
2025.0,0.045,0.086,0.049


## 3. ¿La mejora la sostienen muchos centros o unos pocos?

Un modelo puede mejorar el WAPE agregado porque acierta en unos pocos centros grandes y empeorar
en la mayoría. `compare_to_naive` cuenta en cuántos pares y en cuántas series tiene menos error
absoluto que el ingenuo.

In [5]:
display(pd.DataFrame({m: compare_to_naive(forecasts, m) for m in ("drift", "moving_average")}).round(3))

,drift,moving_average
n_pairs,761.000,761.000
share_pairs_better,0.486,0.271
share_pairs_equal,0.009,0.018
n_series,110.000,110.000
n_series_better,43.000,13.000


## 4. Por tamaño de centro

Terciles de la matrícula en el último curso (`size_last`). Solo descriptivo: el tramo usa un dato de
2025-26 para agrupar, no para prever.

In [6]:
tiers = add_size_tier(forecasts, quality)
by_tier = compute_metrics(tiers, by=["size_tier"])
display(by_tier[["size_tier", "model", "n_pairs", "mae", "wape", "mape", "bias", "wape_vs_naive"]].round(3))

,size_tier,model,n_pairs,mae,wape,mape,bias,wape_vs_naive
0,grande,drift,294,32.536,0.046,0.049,-13.534,-0.169
1,grande,moving_average,294,65.668,0.092,0.104,-54.652,0.678
2,grande,naive,294,39.133,0.055,0.062,-29.017,0.000
3,mediano,drift,298,15.043,0.039,0.040,-1.530,0.083
4,mediano,moving_average,298,22.425,0.058,0.060,-8.886,0.615
5,mediano,naive,298,13.889,0.036,0.038,-4.681,0.000
6,pequeño,drift,169,10.284,0.046,0.058,3.238,0.102
7,pequeño,moving_average,169,13.604,0.060,0.090,2.191,0.458
8,pequeño,naive,169,9.331,0.041,0.054,1.781,0.000


## 5. Series elegibles sin pares

Series elegibles con exactamente `MIN_TRAIN` cursos: no se pueden evaluar.

In [7]:
without_pairs = sorted(set(quality.loc[quality["eligible"], "series_id"]) - set(forecasts["series_id"]))
print(without_pairs)

['Deusto|48012899', 'Pontificia Comillas|07009045']


## 6. Comprobación contra el pipeline

El pipeline (`evaluate_baselines`) escribe `baseline_forecasts.csv` y `baseline_metrics.csv`. Se
recalculan aquí con las funciones del paquete y deben ser idénticos a los ficheros.

In [8]:
eligible = series[series["series_id"].isin(quality.loc[quality["eligible"], "series_id"])]
forecasts_again = rolling_origin_forecasts(eligible)
metrics_again = metrics_report(forecasts_again)

forecasts_disk = pd.read_csv(out_dir / "baseline_forecasts.csv")
metrics_disk = pd.read_csv(out_dir / "baseline_metrics.csv")
pd.testing.assert_frame_equal(forecasts_again, forecasts_disk, check_dtype=False)
pd.testing.assert_frame_equal(metrics_again, metrics_disk, check_dtype=False)
pd.testing.assert_frame_equal(forecasts, forecasts_again)
pd.testing.assert_frame_equal(metrics, metrics_again)
print("El notebook y el pipeline producen exactamente lo mismo.")

El notebook y el pipeline producen exactamente lo mismo.
